# Tạo embedding CLIP

1. Bật **GPU** và **Internet** trong phần cài đặt của Kaggle.
2. Nếu chọn Flickr30k: gắn dataset `eeshawn/flickr30k` bằng **Add Input**.
3. Đổi một dòng `DATASET` ở ô Config.
4. **Run All**. Smoke chạy trước, lỗi thì dừng. Kết quả nằm trong `/kaggle/working/features`.

Hướng dẫn: `docs/guides/chuan-bi-embedding-clip.md`.


## 1. Config

Đây là dòng duy nhất cần đổi.


In [ ]:
DATASET = "flickr30k"  # or "visual_genome"

## 2. Code

Không sửa ở đây. Bản gốc nằm trong `src/mmxai/`; cập nhật bằng `scripts/sync_notebook_code.py`.


In [ ]:
import subprocess
import sys
from pathlib import Path

if DATASET not in ("visual_genome", "flickr30k"):
    raise ValueError(f"DATASET must be visual_genome or flickr30k, got {DATASET}")

CODE_ROOT = "/kaggle/temp/mmxai_src"
OUTPUT_ROOT = "/kaggle/working/features"
# Outside /kaggle/working, so trial output can never be published with the real result.
SMOKE_OUTPUT_ROOT = "/kaggle/temp/smoke"

for folder in ["mmxai/utils", "mmxai/data", "mmxai/features", "configs"]:
    Path(CODE_ROOT, folder).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, CODE_ROOT)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "open-clip-torch==3.3.0"], check=True
)

In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/utils/config.py
"""Read feature extraction configs from YAML into a typed dataclass (ADR-0007)."""

from dataclasses import MISSING, dataclass, fields
from pathlib import Path

import yaml

VISUAL_GENOME = "visual_genome"
FLICKR30K = "flickr30k"


@dataclass(frozen=True)
class FeatureConfig:
    """All settings of one CLIP feature extraction run."""

    dataset: str
    dataset_version: str
    data_dir: str
    output_root: str
    model: str
    pretrained: str
    device: str
    image_batch_size: int
    text_batch_size: int
    num_workers: int
    max_failed_image_fraction: float
    images_dir: str | None = None
    regions_per_image: int | None = None
    sampling_seed: int | None = None


def validate_feature_config(config: FeatureConfig) -> None:
    """Reject settings that do not make sense for the chosen dataset."""
    if config.dataset == VISUAL_GENOME:
        if config.regions_per_image is None or config.sampling_seed is None:
            raise ValueError("visual_genome needs regions_per_image and sampling_seed")
    elif config.dataset == FLICKR30K:
        if config.regions_per_image is not None or config.sampling_seed is not None:
            raise ValueError("flickr30k uses every caption, so it takes no sampling settings")
    else:
        raise ValueError(f"unsupported dataset: {config.dataset}")


def load_feature_config(path: str | Path) -> FeatureConfig:
    """Load and validate one YAML config. A mistyped key is an error, never ignored."""
    with open(path, encoding="utf-8") as file:
        values = yaml.safe_load(file)

    known_keys = {field.name for field in fields(FeatureConfig)}
    required_keys = {field.name for field in fields(FeatureConfig) if field.default is MISSING}
    unknown_keys = set(values) - known_keys
    missing_keys = required_keys - set(values)
    if unknown_keys:
        raise ValueError(f"unknown config keys: {sorted(unknown_keys)}")
    if missing_keys:
        raise ValueError(f"missing config keys: {sorted(missing_keys)}")

    config = FeatureConfig(**values)
    validate_feature_config(config)
    return config


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/data/records.py
"""Plain records shared by every dataset loader, so the encoder needs no dataset knowledge."""

from dataclasses import dataclass
from pathlib import Path


@dataclass(frozen=True)
class ImageRecord:
    """One image to encode."""

    image_id: int
    path: Path


# slots keeps memory down: Visual Genome holds 5.4 million of these before sampling.
@dataclass(frozen=True, slots=True)
class TextRecord:
    """One text to encode, linked to the image it describes."""

    image_id: int
    text_id: int  # region_id (Visual Genome) or caption_index (Flickr30k)
    text: str


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/data/flickr30k.py
"""Read Flickr30k images and captions from Karpathy's dataset_flickr30k.json."""

import json
from pathlib import Path

from mmxai.data.records import ImageRecord, TextRecord

CAPTION_FILE = "dataset_flickr30k.json"


def read_image_entries(data_dir: Path) -> list[dict]:
    """Return the caption file's image entries, sorted by image id."""
    with open(Path(data_dir) / CAPTION_FILE, encoding="utf-8") as file:
        entries = json.load(file)["images"]
    return sorted(entries, key=lambda entry: image_id_from_filename(entry["filename"]))


def image_id_from_filename(filename: str) -> int:
    """Use the number in the filename: unlike imgid, it also matches other Flickr30k sources."""
    stem = Path(filename).stem
    if not stem.isdigit():
        raise ValueError(f"expected a numeric Flickr30k filename, got {filename}")
    return int(stem)


def load_image_records(data_dir: Path, images_dir: Path | None = None) -> list[ImageRecord]:
    """List images named in the caption file. Other files in the folder are ignored."""
    if images_dir is None:
        images_dir = Path(data_dir) / "images"

    records = []
    for entry in read_image_entries(data_dir):
        image_id = image_id_from_filename(entry["filename"])
        records.append(ImageRecord(image_id, Path(images_dir) / entry["filename"]))
    return records


def load_captions(data_dir: Path) -> list[TextRecord]:
    """List every caption, sorted by image then caption position.

    The file's "split" field is deliberately not read: official splits live in splits/.
    """
    captions = []
    for entry in read_image_entries(data_dir):
        image_id = image_id_from_filename(entry["filename"])
        for caption_index, sentence in enumerate(entry["sentences"]):
            captions.append(TextRecord(image_id, caption_index, sentence["raw"]))
    return captions


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/data/visual_genome.py
"""Read Visual Genome v1.2 images and region descriptions, and sample regions per image."""

import json
from pathlib import Path

import numpy as np

from mmxai.data.records import ImageRecord, TextRecord

IMAGE_FILE = "image_data.json"
REGION_FILE = "region_descriptions.json"


def load_image_records(data_dir: Path, images_dir: Path | None = None) -> list[ImageRecord]:
    """List all images, sorted by id. The url tells which folder holds each image."""
    if images_dir is None:
        images_dir = Path(data_dir) / "images"

    with open(Path(data_dir) / IMAGE_FILE, encoding="utf-8") as file:
        entries = json.load(file)

    records = []
    for entry in entries:
        url_parts = entry["url"].split("/")
        folder = url_parts[-2]
        filename = url_parts[-1]
        records.append(ImageRecord(entry["image_id"], Path(images_dir) / folder / filename))
    return sorted(records, key=lambda record: record.image_id)


def load_region_texts(data_dir: Path) -> list[TextRecord]:
    """List all regions, sorted by image then region id.

    Blank phrases are dropped: the embedding of an empty string is not a query.
    """
    with open(Path(data_dir) / REGION_FILE, encoding="utf-8") as file:
        entries = json.load(file)

    regions = []
    for entry in entries:
        for region in entry["regions"]:
            phrase = region["phrase"].strip()
            if phrase:
                regions.append(TextRecord(region["image_id"], region["region_id"], phrase))
    return sorted(regions, key=lambda region: (region.image_id, region.text_id))


def sample_regions(
    regions: list[TextRecord], regions_per_image: int, seed: int
) -> list[TextRecord]:
    """Pick up to regions_per_image regions per image at random, reproducibly.

    The choice ignores content and happens before any retrieval result exists, so
    it cannot favour a model. Fixed ordering plus one generator makes it repeatable.
    """
    regions_by_image: dict[int, list[TextRecord]] = {}
    for region in sorted(regions, key=lambda region: (region.image_id, region.text_id)):
        regions_by_image.setdefault(region.image_id, []).append(region)

    rng = np.random.default_rng(seed)
    sampled = []
    for image_id in sorted(regions_by_image):
        image_regions = regions_by_image[image_id]
        if len(image_regions) <= regions_per_image:
            sampled.extend(image_regions)
            continue
        chosen = rng.choice(len(image_regions), size=regions_per_image, replace=False)
        # Sorting keeps rows in the documented order: by image, then by region id.
        for position in sorted(chosen.tolist()):
            sampled.append(image_regions[position])
    return sampled


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/features/cache.py
"""Save and load cached embeddings as .npz + meta.json (ADR-0041). Needs numpy only."""

import hashlib
import json
import os
import shutil
from collections.abc import Callable
from pathlib import Path

import numpy as np

from mmxai.utils.config import FeatureConfig

IMAGE_FILE = "image_features.npz"
TEXT_FILE = "text_features.npz"
META_FILE = "meta.json"

# A cache is only valid for the settings that produced it.
META_FIELDS_TO_CHECK = [
    "dataset",
    "dataset_version",
    "model",
    "pretrained",
    "regions_per_image",
    "sampling_seed",
]


def feature_dir(config: FeatureConfig, is_partial: bool) -> Path:
    """Return the output folder. Trial runs get their own folder so they are never published."""
    dataset_folder = config.dataset
    if is_partial:
        dataset_folder = f"{config.dataset}_partial"
    return Path(config.output_root) / dataset_folder / f"{config.model}__{config.pretrained}"


def save_features(
    output_dir: Path,
    image_embeddings: np.ndarray,
    image_ids: np.ndarray,
    text_embeddings: np.ndarray,
    text_image_ids: np.ndarray,
    text_id_name: str,
    text_id_values: np.ndarray,
    meta: dict,
    check: Callable[[Path], None] | None = None,
) -> None:
    """Write one complete feature set. Refuses to overwrite an existing one.

    `check` runs on the finished files before they get their final name, so
    output that fails it is removed and can never be mistaken for a real result.
    """
    output_dir = Path(output_dir)
    if output_dir.exists():
        raise FileExistsError(f"{output_dir} already exists; delete it to extract again")

    # Writing to a temporary folder first means an interrupted run can never
    # leave a folder that looks complete.
    temp_dir = output_dir.with_name(output_dir.name + ".tmp")
    if temp_dir.exists():
        shutil.rmtree(temp_dir)
    temp_dir.mkdir(parents=True)

    np.savez(
        temp_dir / IMAGE_FILE,
        embeddings=np.asarray(image_embeddings, dtype=np.float32),
        image_ids=np.asarray(image_ids, dtype=np.int64),
    )
    text_arrays = {
        "embeddings": np.asarray(text_embeddings, dtype=np.float32),
        "image_ids": np.asarray(text_image_ids, dtype=np.int64),
        text_id_name: np.asarray(text_id_values, dtype=np.int64),
    }
    np.savez(temp_dir / TEXT_FILE, **text_arrays)
    with open(temp_dir / META_FILE, "w", encoding="utf-8") as file:
        json.dump(meta, file, indent=2)

    if check is not None:
        try:
            check(temp_dir)
        except Exception:
            shutil.rmtree(temp_dir)
            raise

    os.replace(temp_dir, output_dir)


def load_meta(output_dir: Path) -> dict:
    """Read meta.json. A folder without it is not a complete feature set."""
    with open(Path(output_dir) / META_FILE, encoding="utf-8") as file:
        return json.load(file)


def load_image_features(output_dir: Path) -> tuple[np.ndarray, np.ndarray]:
    """Return (embeddings, image_ids)."""
    load_meta(output_dir)
    with np.load(Path(output_dir) / IMAGE_FILE, allow_pickle=False) as data:
        return data["embeddings"], data["image_ids"]


def load_text_features(output_dir: Path) -> dict[str, np.ndarray]:
    """Return every array of text_features.npz, keyed by name."""
    load_meta(output_dir)
    with np.load(Path(output_dir) / TEXT_FILE, allow_pickle=False) as data:
        return {name: data[name] for name in data.files}


def check_meta_matches_config(meta: dict, config: FeatureConfig) -> None:
    """Raise ValueError if the cache was made with different settings (ADR-0008)."""
    for field_name in META_FIELDS_TO_CHECK:
        expected = getattr(config, field_name)
        if meta.get(field_name) != expected:
            raise ValueError(
                f"cache has {field_name}={meta.get(field_name)!r}, config has {expected!r}"
            )


def compute_code_hash(package_root: Path, module_paths: list[str]) -> str:
    """Hash the modules that ran, so two runs can be checked to use the same code."""
    digest = hashlib.sha256()
    for module_path in module_paths:
        digest.update(module_path.encode("utf-8"))
        digest.update((Path(package_root) / module_path).read_bytes())
    return digest.hexdigest()


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/features/verify.py
"""Check an extracted feature set before anyone relies on it. Needs numpy only."""

from pathlib import Path

import numpy as np

from mmxai.features.cache import load_image_features, load_meta, load_text_features

NORM_TOLERANCE = 1e-3
CAPTIONS_PER_IMAGE = 5


class FeatureVerificationError(ValueError):
    """Raised with every problem found in a feature set."""


def check_embeddings(name: str, embeddings: np.ndarray) -> list[str]:
    """Cosine similarity is only correct for finite vectors of length 1."""
    if not np.isfinite(embeddings).all():
        return [f"{name} embeddings contain NaN or Inf"]
    norms = np.linalg.norm(embeddings, axis=1)
    if not np.isclose(norms, 1.0, atol=NORM_TOLERANCE).all():
        return [f"{name} embeddings are not unit length"]
    return []


def check_row_counts(
    image_embeddings: np.ndarray, image_ids: np.ndarray, texts: dict, meta: dict
) -> list[str]:
    """Every array in a file must have one row per item, matching meta.json."""
    errors = []
    if len(image_embeddings) != len(image_ids):
        errors.append("image rows differ between embeddings and image_ids")
    if len(image_ids) != meta["num_images"]:
        errors.append(f"num_images in meta is {meta['num_images']}, found {len(image_ids)}")

    num_texts = len(texts["embeddings"])
    for name, values in texts.items():
        if len(values) != num_texts:
            errors.append(f"text rows differ between embeddings and {name}")
    if num_texts != meta["num_texts"]:
        errors.append(f"num_texts in meta is {meta['num_texts']}, found {num_texts}")
    return errors


def check_ids(image_ids: np.ndarray, text_image_ids: np.ndarray) -> list[str]:
    """Each text's image id is the retrieval ground truth, so it must point to a real image."""
    errors = []
    if len(np.unique(image_ids)) != len(image_ids):
        errors.append("image_ids contain duplicate values")
    if not np.isin(text_image_ids, image_ids).all():
        errors.append("some texts point to an unknown image")
    return errors


def check_texts_per_image(text_image_ids: np.ndarray, meta: dict) -> list[str]:
    """Visual Genome has at most k sampled regions; a complete Flickr30k has exactly 5 captions."""
    if len(text_image_ids) == 0:
        return []
    _, counts = np.unique(text_image_ids, return_counts=True)
    if meta["dataset"] == "visual_genome" and counts.max() > meta["regions_per_image"]:
        return [f"some images have more than {meta['regions_per_image']} regions"]
    # A trial run may stop in the middle of an image's captions.
    is_complete_flickr = meta["dataset"] == "flickr30k" and not meta["is_partial"]
    if is_complete_flickr and (counts != CAPTIONS_PER_IMAGE).any():
        return [f"some images do not have exactly {CAPTIONS_PER_IMAGE} captions"]
    return []


def check_alignment(
    image_embeddings: np.ndarray,
    image_ids: np.ndarray,
    text_embeddings: np.ndarray,
    text_image_ids: np.ndarray,
    seed: int,
    num_pairs: int,
) -> list[str]:
    """Correct text-image pairs should score above shuffled pairs.

    This only detects rows that lost their alignment. The result never feeds
    into choosing a setting, so it is not tuning on any split.
    """
    if len(image_ids) < 2 or len(text_image_ids) == 0:
        return []

    row_of_image = {int(image_id): row for row, image_id in enumerate(image_ids)}
    rng = np.random.default_rng(seed)
    sample_size = min(num_pairs, len(text_image_ids))
    text_rows = rng.choice(len(text_image_ids), size=sample_size, replace=False)
    matched_rows = np.array([row_of_image[int(text_image_ids[row])] for row in text_rows])
    shuffled_rows = rng.permutation(matched_rows)

    # A shuffle can leave a pair unchanged, which says nothing, so compare only changed pairs.
    changed = shuffled_rows != matched_rows
    if not changed.any():
        return []

    samples = text_embeddings[text_rows[changed]]
    matched_score = np.sum(samples * image_embeddings[matched_rows[changed]], axis=1).mean()
    shuffled_score = np.sum(samples * image_embeddings[shuffled_rows[changed]], axis=1).mean()
    if matched_score <= shuffled_score:
        return [
            "text and image rows look misaligned: "
            f"matched cosine {matched_score:.4f}, shuffled cosine {shuffled_score:.4f}"
        ]
    return []


def verify_features(output_dir: Path, seed: int = 0, num_pairs: int = 1000) -> None:
    """Raise FeatureVerificationError listing every problem found in output_dir."""
    meta = load_meta(output_dir)
    image_embeddings, image_ids = load_image_features(output_dir)
    texts = load_text_features(output_dir)
    text_embeddings = texts["embeddings"]
    text_image_ids = texts["image_ids"]

    errors = check_row_counts(image_embeddings, image_ids, texts, meta)
    errors += check_embeddings("image", image_embeddings)
    errors += check_embeddings("text", text_embeddings)
    errors += check_ids(image_ids, text_image_ids)
    errors += check_texts_per_image(text_image_ids, meta)

    # The alignment check looks rows up by id, so it needs the checks above to have passed.
    if not errors:
        errors += check_alignment(
            image_embeddings, image_ids, text_embeddings, text_image_ids, seed, num_pairs
        )

    if errors:
        raise FeatureVerificationError("; ".join(errors))


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/features/clip_encoder.py
"""Encode images and texts with a frozen CLIP model. CLIP is never trained (PDF section 4.1)."""

from collections.abc import Callable
from dataclasses import dataclass

import numpy as np
import torch
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from mmxai.data.records import ImageRecord

PROGRESS_EVERY_BATCHES = 100


@dataclass
class ClipBundle:
    """The frozen model together with the exact preprocessing it was trained with."""

    model: torch.nn.Module
    preprocess: Callable
    tokenizer: Callable
    device: str


def freeze_model(model: torch.nn.Module) -> torch.nn.Module:
    """Switch to inference mode and make sure no weight can change."""
    model.eval()
    for parameter in model.parameters():
        parameter.requires_grad = False
    return model


def resolve_device(device: str) -> str:
    """Turn "auto" into a real device name."""
    if device != "auto":
        return device
    if torch.cuda.is_available():
        return "cuda"
    return "cpu"


def load_frozen_clip(model_name: str, pretrained: str, device: str) -> ClipBundle:
    """Load CLIP weights and preprocessing from open_clip."""
    # Imported here so tests with a fake model do not need open_clip.
    import open_clip

    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
    tokenizer = open_clip.get_tokenizer(model_name)
    real_device = resolve_device(device)
    model = freeze_model(model).to(real_device)
    return ClipBundle(model, preprocess, tokenizer, real_device)


def library_versions() -> dict[str, str]:
    """Record the versions that produced the embeddings."""
    import open_clip

    return {"open_clip": open_clip.__version__, "torch": torch.__version__}


class ImageRecordDataset(Dataset):
    """Reads images in DataLoader workers, because decoding is the slow part."""

    def __init__(self, records: list[ImageRecord], preprocess: Callable) -> None:
        self.records = records
        self.preprocess = preprocess

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[int, torch.Tensor | None]:
        record = self.records[index]
        # One bad file must not end a run of 100,000 images. The caller counts the
        # failures and stops if there are too many.
        try:
            with Image.open(record.path) as image:
                return record.image_id, self.preprocess(image.convert("RGB"))
        except (OSError, ValueError):
            return record.image_id, None


def collate_images(
    batch: list[tuple[int, torch.Tensor | None]],
) -> tuple[torch.Tensor | None, list[int], list[int]]:
    """Separate readable images from failed ones. Returns (images, image_ids, failed_ids)."""
    tensors = []
    image_ids = []
    failed_ids = []
    for image_id, tensor in batch:
        if tensor is None:
            failed_ids.append(image_id)
        else:
            tensors.append(tensor)
            image_ids.append(image_id)

    if not tensors:
        return None, image_ids, failed_ids
    return torch.stack(tensors), image_ids, failed_ids


def normalize(features: torch.Tensor) -> np.ndarray:
    """Scale to length 1 so a dot product equals cosine similarity."""
    features = features.float()
    features = features / features.norm(dim=-1, keepdim=True)
    return features.cpu().numpy().astype(np.float32)


def join_batches(batches: list[np.ndarray]) -> np.ndarray:
    """Join batch results. np.concatenate cannot handle an empty list."""
    if not batches:
        return np.zeros((0, 0), dtype=np.float32)
    return np.concatenate(batches, axis=0)


def encode_images(
    bundle: ClipBundle, records: list[ImageRecord], batch_size: int, num_workers: int
) -> tuple[np.ndarray, np.ndarray, list[int]]:
    """Return (embeddings, image_ids that were encoded, failed image_ids), in input order."""
    dataset = ImageRecordDataset(records, bundle.preprocess)
    # shuffle=False keeps rows aligned with the ids we record.
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        collate_fn=collate_images,
    )

    batches = []
    all_image_ids = []
    all_failed_ids = []
    with torch.no_grad():
        for batch_number, (images, image_ids, failed_ids) in enumerate(loader, start=1):
            all_failed_ids.extend(failed_ids)
            if images is not None:
                features = bundle.model.encode_image(images.to(bundle.device))
                batches.append(normalize(features))
                all_image_ids.extend(image_ids)
            if batch_number % PROGRESS_EVERY_BATCHES == 0:
                print(f"images: {len(all_image_ids)} encoded, {len(all_failed_ids)} failed")

    return join_batches(batches), np.array(all_image_ids, dtype=np.int64), all_failed_ids


def encode_texts(bundle: ClipBundle, texts: list[str], batch_size: int) -> np.ndarray:
    """Return one embedding per text, in input order."""
    batches = []
    with torch.no_grad():
        for start in range(0, len(texts), batch_size):
            tokens = bundle.tokenizer(texts[start : start + batch_size]).to(bundle.device)
            batches.append(normalize(bundle.model.encode_text(tokens)))
            batch_number = len(batches)
            if batch_number % PROGRESS_EVERY_BATCHES == 0:
                print(f"texts: {start + batch_size} encoded")
    return join_batches(batches)


In [ ]:
%%writefile /kaggle/temp/mmxai_src/mmxai/features/pipeline.py
"""Connect the extraction steps: records -> embeddings -> saved, verified feature set."""

from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

from mmxai.data import flickr30k, visual_genome
from mmxai.data.records import ImageRecord, TextRecord
from mmxai.features.cache import compute_code_hash, feature_dir, save_features
from mmxai.features.verify import verify_features
from mmxai.utils.config import FLICKR30K, VISUAL_GENOME, FeatureConfig

# The modules whose content decides the output. Fixed order keeps the hash stable.
# Notebooks carry a copy of exactly these files (ADR-0041).
CODE_MODULES = [
    "utils/config.py",
    "data/records.py",
    "data/flickr30k.py",
    "data/visual_genome.py",
    "features/cache.py",
    "features/verify.py",
    "features/clip_encoder.py",
    "features/pipeline.py",
]

TEXT_SOURCES = {VISUAL_GENOME: "region_descriptions", FLICKR30K: "karpathy_captions"}
TEXT_ID_NAMES = {VISUAL_GENOME: "text_ids", FLICKR30K: "caption_index"}


def with_path_overrides(
    config: FeatureConfig,
    data_dir: str | None,
    images_dir: str | None,
    output_root: str | None,
) -> FeatureConfig:
    """Apply command line paths, which differ between a laptop and Kaggle."""
    if data_dir is not None:
        config = replace(config, data_dir=data_dir)
    if images_dir is not None:
        config = replace(config, images_dir=images_dir)
    if output_root is not None:
        config = replace(config, output_root=output_root)
    return config


def load_dataset_records(config: FeatureConfig) -> tuple[list[ImageRecord], list[TextRecord]]:
    """Return the images and texts to encode for the configured dataset."""
    data_dir = Path(config.data_dir)
    images_dir = None
    if config.images_dir is not None:
        images_dir = Path(config.images_dir)

    if config.dataset == VISUAL_GENOME:
        images = visual_genome.load_image_records(data_dir, images_dir)
        regions = visual_genome.load_region_texts(data_dir)
        texts = visual_genome.sample_regions(
            regions, config.regions_per_image, config.sampling_seed
        )
        return images, texts

    images = flickr30k.load_image_records(data_dir, images_dir)
    texts = flickr30k.load_captions(data_dir)
    return images, texts


def filter_texts_for_images(texts: list[TextRecord], kept_image_ids: set[int]) -> list[TextRecord]:
    """Keep only texts whose image exists, so every text has a valid retrieval target."""
    return [text for text in texts if text.image_id in kept_image_ids]


def apply_limits(
    images: list[ImageRecord],
    texts: list[TextRecord],
    limit_images: int | None,
    limit_texts: int | None,
) -> tuple[list[ImageRecord], list[TextRecord]]:
    """Cut the data down for a trial run."""
    if limit_images is not None:
        images = images[:limit_images]
        kept_image_ids = {image.image_id for image in images}
        texts = filter_texts_for_images(texts, kept_image_ids)
    if limit_texts is not None:
        texts = texts[:limit_texts]
    return images, texts


def check_failed_fraction(num_failed: int, num_total: int, max_fraction: float) -> None:
    """A few broken images are normal; many mean the download or the paths are wrong."""
    if num_failed > num_total * max_fraction:
        raise ValueError(
            f"{num_failed} of {num_total} images could not be read "
            f"(limit is {max_fraction:.2%}); check the download and images_dir"
        )


def build_meta(
    config: FeatureConfig,
    num_images: int,
    num_texts: int,
    embedding_dim: int,
    failed_image_ids: list[int],
    is_partial: bool,
    versions: dict[str, str],
    code_hash: str,
) -> dict:
    """Describe how a feature set was made, for checks and for the oral exam."""
    return {
        "dataset": config.dataset,
        "dataset_version": config.dataset_version,
        "model": config.model,
        "pretrained": config.pretrained,
        "embedding_dim": embedding_dim,
        "normalized": True,
        "num_images": num_images,
        "num_texts": num_texts,
        "text_source": TEXT_SOURCES[config.dataset],
        "regions_per_image": config.regions_per_image,
        "sampling_seed": config.sampling_seed,
        "failed_image_ids": failed_image_ids,
        "is_partial": is_partial,
        "open_clip_version": versions["open_clip"],
        "torch_version": versions["torch"],
        "code_hash": code_hash,
        "created_at": datetime.now(UTC).isoformat(timespec="seconds"),
    }


def run_extraction(
    config: FeatureConfig, limit_images: int | None = None, limit_texts: int | None = None
) -> Path:
    """Run one extraction and return the folder holding the verified result."""
    # Imported here because everything else in this module must work without torch.
    from mmxai.features import clip_encoder

    is_partial = limit_images is not None or limit_texts is not None
    output_dir = feature_dir(config, is_partial)
    # Checked first so a mistake costs seconds, not a whole encoding run.
    if output_dir.exists():
        raise FileExistsError(f"{output_dir} already exists; delete it to extract again")

    images, texts = load_dataset_records(config)
    images, texts = apply_limits(images, texts, limit_images, limit_texts)
    print(f"{config.dataset}: {len(images)} images, {len(texts)} texts")

    bundle = clip_encoder.load_frozen_clip(config.model, config.pretrained, config.device)
    # A forgotten GPU setting otherwise only shows up as a run that takes hours.
    print(f"device: {bundle.device}")
    image_embeddings, image_ids, failed_ids = clip_encoder.encode_images(
        bundle, images, config.image_batch_size, config.num_workers
    )
    check_failed_fraction(len(failed_ids), len(images), config.max_failed_image_fraction)

    texts = filter_texts_for_images(texts, set(image_ids.tolist()))
    text_embeddings = clip_encoder.encode_texts(
        bundle, [text.text for text in texts], config.text_batch_size
    )

    package_root = Path(__file__).resolve().parent.parent
    meta = build_meta(
        config,
        num_images=len(image_ids),
        num_texts=len(texts),
        embedding_dim=image_embeddings.shape[1],
        failed_image_ids=failed_ids,
        is_partial=is_partial,
        versions=clip_encoder.library_versions(),
        code_hash=compute_code_hash(package_root, CODE_MODULES),
    )
    save_features(
        output_dir,
        image_embeddings=image_embeddings,
        image_ids=image_ids,
        text_embeddings=text_embeddings,
        text_image_ids=[text.image_id for text in texts],
        text_id_name=TEXT_ID_NAMES[config.dataset],
        text_id_values=[text.text_id for text in texts],
        meta=meta,
        check=verify_features,
    )
    print(f"saved and verified: {output_dir}")
    return output_dir


In [ ]:
%%writefile /kaggle/temp/mmxai_src/configs/features_visual_genome.yaml
# CLIP feature extraction for Visual Genome (see docs/guides/chuan-bi-embedding-clip.md).
dataset: visual_genome
dataset_version: "1.2"
data_dir: data/visual_genome
output_root: artifacts/features

model: ViT-B-32
pretrained: laion2b_s34b_b79k
device: auto            # cuda if available, otherwise cpu
image_batch_size: 256
text_batch_size: 1024
num_workers: 4
max_failed_image_fraction: 0.001

# Fixed before any retrieval result exists, so the choice cannot favour a model.
regions_per_image: 5
sampling_seed: 0


In [ ]:
%%writefile /kaggle/temp/mmxai_src/configs/features_flickr30k.yaml
# CLIP feature extraction for Flickr30k (see docs/guides/chuan-bi-embedding-clip.md).
dataset: flickr30k
dataset_version: karpathy
data_dir: data/flickr30k
output_root: artifacts/features

model: ViT-B-32
pretrained: laion2b_s34b_b79k
device: auto            # cuda if available, otherwise cpu
image_batch_size: 256
text_batch_size: 1024
num_workers: 4
max_failed_image_fraction: 0.001


In [ ]:
import shutil

from mmxai.features.pipeline import run_extraction, with_path_overrides
from mmxai.utils.config import load_feature_config

VISUAL_GENOME_DIR = Path("/kaggle/temp/visual_genome")
FLICKR_DIR = Path("/kaggle/temp/flickr30k")
BASE_URL = "https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset"
VISUAL_GENOME_ZIPS = [
    ("https://cs.stanford.edu/people/rak248/VG_100K_2/images.zip", "images"),
    ("https://cs.stanford.edu/people/rak248/VG_100K_2/images2.zip", "images"),
    (f"{BASE_URL}/image_data.json.zip", "."),
    (f"{BASE_URL}/region_descriptions.json.zip", "."),
]
CAPTION_URL = "https://cs.stanford.edu/people/karpathy/deepimagesent/caption_datasets.zip"


def download_and_unzip(url, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    archive = target_dir / url.split("/")[-1]
    subprocess.run(["curl", "-L", "--fail", "-C", "-", "-o", str(archive), url], check=True)
    subprocess.run(["unzip", "-q", "-o", str(archive), "-d", str(target_dir)], check=True)
    archive.unlink()  # Free disk space: the temporary folder is limited too.


def prepare_visual_genome():
    if not (VISUAL_GENOME_DIR / "region_descriptions.json").exists():
        for url, folder in VISUAL_GENOME_ZIPS:
            download_and_unzip(url, VISUAL_GENOME_DIR / folder)
    return VISUAL_GENOME_DIR, VISUAL_GENOME_DIR / "images"


def find_flickr_images():
    for folder in Path("/kaggle/input").rglob("flickr30k_images"):
        if (folder / "1000092795.jpg").exists():
            return folder
    raise FileNotFoundError("Attach the Kaggle dataset eeshawn/flickr30k with Add Input")


def prepare_flickr30k():
    images_dir = find_flickr_images()
    if not (FLICKR_DIR / "dataset_flickr30k.json").exists():
        download_and_unzip(CAPTION_URL, FLICKR_DIR)
    return FLICKR_DIR, images_dir


def make_config(dataset, output_root):
    if dataset == "visual_genome":
        data_dir, images_dir = prepare_visual_genome()
    else:
        data_dir, images_dir = prepare_flickr30k()
    config = load_feature_config(f"{CODE_ROOT}/configs/features_{dataset}.yaml")
    return with_path_overrides(config, str(data_dir), str(images_dir), output_root)

## 3. Smoke

Chạy thử 1.000 ảnh + 10.000 văn bản. Lỗi ở đây thì dừng, không chạy phần đầy đủ.


In [ ]:
shutil.rmtree(SMOKE_OUTPUT_ROOT, ignore_errors=True)
smoke_config = make_config(DATASET, SMOKE_OUTPUT_ROOT)
smoke_dir = run_extraction(smoke_config, limit_images=1000, limit_texts=10000)
print((smoke_dir / "meta.json").read_text())

## 4. Dataset 1: Visual Genome

Chỉ chạy khi `DATASET = "visual_genome"`.


In [ ]:
if DATASET == "visual_genome":
    output_dir = run_extraction(make_config("visual_genome", OUTPUT_ROOT))
    print((output_dir / "meta.json").read_text())
else:
    print("Skipped: DATASET is", DATASET)

## 5. Dataset 2: Flickr30k

Chỉ chạy khi `DATASET = "flickr30k"`.


In [ ]:
if DATASET == "flickr30k":
    output_dir = run_extraction(make_config("flickr30k", OUTPUT_ROOT))
    print((output_dir / "meta.json").read_text())
else:
    print("Skipped: DATASET is", DATASET)